# TATTVA-SWARM · Benchmark de capacidad (reproducible)

**Artefacto del lab** — `~/lab-master/notebooks/tattva_swarm_capacity_benchmark.ipynb`
Autoría: agente OpenCode (misión TATTVA-SWARM, 2026-10-18) · Estado: READY_FOR_HUMAN_APPROVAL (la sección opcional pesada en Colab requiere aprobación humana).
Notebook *autocontenido*: no clona repos inaccesibles ni requiere credenciales.

## 1. Resumen ejecutivo

- **Qué mide:** capacidad *lógica* del swarm (cola de tareas, workers concurrentes, idempotencia, cancelación, reanudación) y el baseline de cómputo del runtime donde correría.
- **Qué NO mide:** no ejecuta "108M de procesos". El techo 108,000,000 de Meru-Piṅgala es un **espacio combinatorio** (lattice `C(30,6)=593,775` verificado; `fortune_max_workers=108,000,000` como cota lógica), no concurrencia física.
- **Ya medido localmente (Etapa A, host del operador):** 100/100 tareas, 0 doble-claim, cancelación y reanudación OK, ~0.82 tareas/s con 4 workers (cuello de botella: ~2 s por operación de BD vía `docker exec` en host cargado). Ver `~/lab-master/reports/tattva-swarm/benchmark_stage_a_results.json` y `worker-capacity.md`.
- **Este notebook:** reproduce el baseline en Colab (T4) con presupuesto mínimo y deja preparada, con guards, la corrida opcional del binario nativo.

## 2. Objetivo y alcance

- **Objetivo:** benchmark reproducible que mida (a) baseline CPU/GPU del runtime; (b) throughput de una cola lógica con workers concurrentes; (c) [opcional, con GATE] capacidad del binario nativo `shiva` con N acotado.
- **Alcance excluido:** no toca servidores de terceros; no usa credenciales; no descarga datos personales; no envía nada a HackerOne.
- **Criterio de honestidad:** cada celda imprime lo que mide; si algo no está disponible (p. ej. GPU), lo declara y continúa en modo CPU.

## 3. Supuestos y requisitos

- Runtime Colab con Python ≥ 3.11 y, si es posible, GPU T4 preasignada (evidencia real del entorno del operador: Tesla T4, torch 2.11.0+cu130, CUDA=True).
- RAM ≥ 12 GB solo para la sección 9 (pesada). Las secciones 1–8 corren con cualquier runtime.
- **Sin claves en el notebook:** cualquier backend (p. ej. NIM) se configura por variables de entorno del runtime, nunca en celdas ni outputs.
- Datos 100 % sintéticos (se generan en la celda).

## 4. Límites de capacidad conocidos

| capa | límite real verificado | fuente |
|---|---|---|
| Host del operador | 4 cores · ~6.7 GB RAM · load saturado | preflight.md |
| Cola lógica (Etapa A) | ~0.82 tareas/s con 4 workers (2 s por op de BD vía `docker exec`) | benchmark_stage_a_results.json |
| Intra-proceso | el techo físico es de cores/RAM; no hay límite de "workers" en Python más allá de eso | — |
| Espacio combinatorio Meru | lattice `C(30,6)=593,775`; cota lógica `108,000,000` | preflight.md §4 |
| Colab T4 (evidencia) | ~12–16 GB RAM · ~16 GB VRAM | celdas de entorno ya ejecutadas en el notebook del operador |

**Regla dura:** nada de aquí autoriza prometer "108M concurrentes"; es el mapeo lógico del espacio, no concurrencia física.

## 5. Arquitectura (vista de capacidad)

```
router shiva (:9620, contrato jahyah) ── decisión/registro ──▶ PG shiva_moa (moa.*)
   │
   ├─ workers lógicos (cola PG `FOR UPDATE SKIP LOCKED`; Etapa A) ─▶ results JSON
   ├─ binario nativo `shiva` (Rust; tattva-benchmark / tattva-evaluate) ─▶ local o Colab
   └─ kanban (hermes) + grants/policy ─▶ gates humanos (aprobación antes de cualquier prueba)
```

## 6. Procedencia de modelos y datos

- **Router:** tiers `small/normal/large/adaptation/research`; backends por entorno; por defecto `none-echo` determinista (verificado en el flow test local: decisión `5559c460-…`, idempotente, sin contenido inventado).
- **Modelos NIM (opcional):** los configura el operador por variables de entorno del runtime; **jamás** claves dentro del notebook. Si se usan, medir `reasoning_tokens` (en la prueba real: 89 de 106 tokens completados fueron razonamiento).
- **Datos:** sintéticos (in-notebook). Casos reales: solo con procedencia verificable (política H1 del lab: nada inventado).

## 7. Versiones y pinning

- Python del runtime; `numpy`/`torch` preinstalados (visto: torch 2.11.0+cu130 en T4).
- Nada más se instala por defecto. El binario nativo (opcional) se compila desde un tarball que **sube el operador** (no se descarga de repos inaccesibles).
- Registrar al correr: `platform.python_version()`, `numpy.__version__`, `torch.version.cuda` (la celda 1 lo imprime).

## 8. Pasos reproducibles (orden y presupuesto)

1. Celda de entorno — 5 s. Declara GPU/RAM/versiones.
2. Micro-bench CPU — ~10 s.
3. Micro-bench GPU (si hay) — ~10 s.
4. Cola lógica 4 workers × 2,000 tareas — ~30–60 s.
5. (Opcional, **GATE**) `shiva tattva-benchmark N` con N ∈ {1M, 10M, 54M}; incluye guard de RAM y no corre sin tarball subido.

> No ejecutar la sección 9 sin aprobación humana y sin revisar `free -h`.

In [ ]:
# Celda 1 — entorno (sin secretos). ~5 s
import platform, shutil, subprocess
print("python:", platform.python_version(), "| host:", platform.node())
try:
    import numpy, torch
    print("numpy:", numpy.__version__)
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
except Exception as e:
    print("libs:", e)
g = shutil.which("nvidia-smi") or "/opt/bin/nvidia-smi"
try:
    out = subprocess.run([g, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", out or "(sin GPU)")
except Exception as e:
    print("gpu:", e)
print(open("/proc/meminfo").readline().strip())

In [ ]:
# Celda 2 — micro-bench CPU (matmul float32 1200^3). ~10 s
import numpy as np, time
rng = np.random.default_rng(0)
a = rng.random((1200, 1200), dtype=np.float32)
t0 = time.perf_counter(); _ = a @ a.T; dt = time.perf_counter() - t0
print(f"CPU matmul 1200x1200 float32: {dt:.3f} s -> ~{2 * 1200**3 / dt / 1e9:.2f} GFLOP/s (PASS > 10)")

In [ ]:
# Celda 3 — micro-bench GPU (si hay). ~10 s
try:
    import torch, time
    if torch.cuda.is_available():
        x = torch.rand((4096, 4096), device="cuda")
        torch.cuda.synchronize(); t0 = time.perf_counter()
        y = x @ x
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        print(f"GPU matmul 4096x4096 fp32: {dt*1000:.1f} ms -> ~{2 * 4096**3 / dt / 1e12:.2f} TFLOP/s (PASS > 1)")
    else:
        print("sin GPU disponible: se omite (modo CPU)")
except Exception as e:
    print("gpu bench:", e)

In [ ]:
# Celda 4 — cola lógica: 4 workers x 2,000 tareas sintéticas (semántica de cola, en memoria). ~30-60 s
import threading, time, hashlib, queue
N, W = 2000, 4
q = queue.Queue()
for i in range(N):
    q.put(f"t-{i:04d}")
done = [0]; lock = threading.Lock(); errs = []

def worker():
    h = b"seed"
    try:
        while True:
            try:
                key = q.get_nowait()
            except queue.Empty:
                break
            for _ in range(4000):
                h = hashlib.sha256(h).digest()
            with lock:
                done[0] += 1
    except Exception as e:
        errs.append(repr(e))

t0 = time.perf_counter()
ts = [threading.Thread(target=worker) for _ in range(W)]
[t.start() for t in ts]; [t.join() for t in ts]
dt = time.perf_counter() - t0
print(f"{done[0]}/{N} tareas en {dt:.2f} s -> {done[0]/dt:.1f} tareas/s ({W} workers)")
print("errores:", errs)
print("referencia Etapa A local (con coste de BD ~2 s/op): 0.82 tareas/s")

## 9. (Opcional · GATE) binario nativo `shiva` — capacidad acotada

Requiere que el operador suba `/content/shiva-native.tar.gz` (contenido de `shiva-swarm/native`).
Escalar N manualmente: `1_000_000` → `10_000_000` → `54_000_000`. **108M requiere ~14 GB y no se recomienda en Colab.**
La celda incluye guard de RAM y falla limpio si no hay tarball.

In [ ]:
# Celda 5 — OPCIONAL y con GATE HUMANO. NO correr sin aprobación.
import os, subprocess, tarfile, shutil
N = 1_000_000  # escalar manualmente: 1M -> 10M -> 54M
page = os.sysconf("SC_PAGE_SIZE"); pages = os.sysconf("SC_AVPHYS_PAGES")
free_gb = page * pages / 1e9
print(f"RAM libre: {free_gb:.1f} GB · N solicitado: {N:,}")
if N >= 50_000_000 and free_gb < 11:
    raise SystemExit("Guard: RAM insuficiente para N grande; reducir N o cambiar de runtime.")
if not os.path.exists("/content/shiva-native.tar.gz"):
    print("Falta /content/shiva-native.tar.gz (subirlo manualmente). Se omite.")
else:
    shutil.rmtree("/content/shiva-two", ignore_errors=True)
    os.makedirs("/content/shiva-two")
    with tarfile.open("/content/shiva-native.tar.gz") as tf:
        tf.extractall("/content/shiva-two")
    subprocess.run(["bash", "-lc",
                    "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"],
                   check=False)
    r = subprocess.run(["bash", "-lc",
                        "cd /content/shiva-two/native && $HOME/.cargo/bin/cargo build --release -j2 2>&1 | tail -3"],
                       text=True, capture_output=True)
    print(r.stdout[-800:])
    r2 = subprocess.run(["bash", "-lc",
                         f"cd /content/shiva-two && ./native/target/release/shiva tattva-benchmark {N}"],
                        text=True, capture_output=True)
    print(r2.stdout[-2000:])
    print("rc:", r2.returncode)

## 10. Resultados esperados y criterios de aceptación

| medición | esperado (orden de magnitud) | PASS si |
|---|---|---|
| CPU matmul | 50–150 GFLOP/s (Colab CPU) | > 10 GFLOP/s |
| GPU matmul (T4) | 2–5 TFLOP/s fp32 | > 1 TFLOP/s |
| Cola lógica 4w/2k | 300–1,500 tareas/s (en memoria) | ≥ 100 tareas/s y 0 errores |
| Nativo 1M (opcional) | corrida limpia con guard de RAM | rc = 0 |

La cola lógica en memoria es ~1000× más rápida que la Etapa A local porque elimina el costo de ~2 s por operación de BD (ver `worker-capacity.md`). Eso también demuestra que el límite estaba en el *driver*, no en el modelo de workers.

## 11. Coste, almacenamiento, riesgos y checklist

- **Coste:** secciones 1–8 < 2 min de cómputo. La sección 9 compila Rust (~5–10 min CPU) y corre N a elección del operador.
- **Almacenamiento:** resultados < 1 MB; limpiar al terminar (`shutil.rmtree("/content/shiva-two")`).
- **Riesgos:** OOM en N grandes (guard + fallback), cuota agotada (correr liviano primero), kernel reiniciado (ejecutar celdas en orden), T4 no asignada (todo tiene modo CPU).

**Checklist antes de ejecutar:**
- [ ] Ninguna celda contiene claves/tokens/datos personales.
- [ ] Secciones 1–8 sin GATE; sección 9 solo con aprobación humana.
- [ ] `free -h` revisado antes de N ≥ 50M.
- [ ] Resultados copiados a `~/lab-master/reports/tattva-swarm/` al terminar.
- [ ] Cero tráfico contra dominios de terceros (solo lo que el runtime requiera para herramientas).

---
**Referencias del lab:** `~/lab-master/reports/tattva-swarm/{preflight.md, task-graph.json, worker-capacity.md, benchmark_stage_a_results.json}` ·
`~/lab-master/evidence/router_flow_test_20261018.txt` · fuente del notebook: `~/lab-master/work/build_capacity_notebook.py`.